In [12]:
#!pip install openpyxl

- 1. device_objects.py → 디바이스별 Skill.xxx 추출
- 2. service_objects.py → 각 스킬의 Values, Functions, Enums 파싱
- 3. full_name, argument_type, return_type, descriptor 등 열 정리
- 4. device_skill_action_registry_output.xlsx 파일로 저장

In [3]:
import ast
import re
import pandas as pd
from pathlib import Path
from typing import Tuple, List


In [13]:
def parse_device_skills_ast(code: str) -> dict:
    tree = ast.parse(code)
    device_skills = {}
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            device = node.name
            for stmt in node.body:
                if isinstance(stmt, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "skills" for t in stmt.targets):
                    skills = []
                    if isinstance(stmt.value, ast.Set):
                        for elt in stmt.value.elts:
                            if isinstance(elt, ast.Attribute):
                                skills.append(elt.attr)
                    device_skills[device] = skills
    return device_skills

def extract_descriptor(funcs) -> str:
    for func in funcs:
        if isinstance(func, ast.FunctionDef) and func.name == "descriptor":
            for node in ast.walk(func):
                if isinstance(node, ast.Str):
                    return node.s.strip()
    return ""

def extract_type_format(funcs) -> Tuple[str, str]:
    for func in funcs:
        if isinstance(func, ast.FunctionDef) and func.name in ("value_type", "argument_type", "return_type"):
            for node in ast.walk(func):
                if isinstance(node, ast.Call) and getattr(node.func, "id", "") == "MXValueType":
                    type_name = ""
                    fmt = ""
                    if len(node.args) >= 1:
                        if isinstance(node.args[0], ast.Attribute):
                            type_name = node.args[0].attr
                    if len(node.args) == 2:
                        if isinstance(node.args[1], ast.Attribute):
                            fmt = node.args[1].attr
                        elif isinstance(node.args[1], ast.Name):
                            fmt = node.args[1].id
                    return type_name, fmt
    return "", ""

def parse_service_ast(code: str) -> List[dict]:
    tree = ast.parse(code)
    rows = []
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            skill = node.name
            descriptor = extract_descriptor(node.body)
            enum_desc = ""
            for item in node.body:
                if isinstance(item, ast.ClassDef) and item.name == "Enums":
                    enum_desc = extract_descriptor(item.body)
                if isinstance(item, ast.ClassDef) and item.name == "Values":
                    for sub in item.body:
                        if isinstance(sub, ast.ClassDef):
                            val_name = sub.name
                            ret_desc = extract_descriptor(sub.body)
                            rtype, rformat = extract_type_format(sub.body)
                            rows.append({
                                "skill": skill,
                                "action": val_name,
                                "type": "value",
                                "descriptor": descriptor,
                                "return_descriptor": ret_desc,
                                "argument_descriptor": "",
                                "argument_type": "",
                                "argument_format": "",
                                "argument_bounds": "",
                                "return_type": rtype,
                                "return_format": rformat,
                                "return_bounds": "",
                                "enums_descriptor": enum_desc,
                            })
                if isinstance(item, ast.ClassDef) and item.name == "Functions":
                    for sub in item.body:
                        if isinstance(sub, ast.ClassDef):
                            fn_name = sub.name
                            arg_desc = extract_descriptor(sub.body)
                            rtype, rformat = extract_type_format(sub.body)
                            args = []
                            for argblock in sub.body:
                                if isinstance(argblock, ast.ClassDef) and argblock.name == "Arguments":
                                    for arg in argblock.body:
                                        if isinstance(arg, ast.ClassDef):
                                            atype, aformat = extract_type_format(arg.body)
                                            abounds = extract_descriptor(arg.body)
                                            args.append((atype, aformat, abounds))
                            rows.append({
                                "skill": skill,
                                "action": fn_name,
                                "type": "function",
                                "descriptor": descriptor,
                                "return_descriptor": "",
                                "argument_descriptor": arg_desc,
                                "argument_type": " | ".join(a[0] for a in args),
                                "argument_format": " | ".join(a[1] for a in args),
                                "argument_bounds": " | ".join(a[2] for a in args),
                                "return_type": rtype or "VOID",
                                "return_format": rformat,
                                "return_bounds": "",
                                "enums_descriptor": enum_desc,
                            })
    return rows

In [6]:
def generate_device_skill_table(device_path: str, service_path: str, output_excel_path: str) -> pd.DataFrame:
    device_code = Path(device_path).read_text()
    service_code = Path(service_path).read_text()
    device_skills = parse_device_skills_ast(device_code)
    service_rows = parse_service_ast(service_code)

    full_entries = []
    for device, skills in device_skills.items():
        for row in service_rows:
            if row["skill"] in skills:
                full_entries.append({
                    "full_name": f"#{device}.{row['skill']}_{row['action']}",
                    "device": device,
                    "skill": row["skill"],
                    **row
                })

    df = pd.DataFrame(full_entries)
    df.to_excel(output_excel_path, index=False)
    return df


In [14]:
import os

if __name__ == "__main__":
    root_path = '../datasets'
    d_p = os.path.join(root_path,"device_objects.py")
    s_p = os.path.join(root_path,"service_objects.py")
    df = generate_device_skill_table(d_p, s_p, "device_skill_action_registry_output.xlsx")
    print(df.head(10))


/tmp/ipykernel_741231/2650427379.py:21: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Str):
/tmp/ipykernel_741231/2650427379.py:22: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  return node.s.strip()


                                           full_name          device  \
0  #AirConditioner.airConditionerMode_airConditio...  AirConditioner   
1  #AirConditioner.airConditionerMode_targetTempe...  AirConditioner   
2  #AirConditioner.airConditionerMode_supportedAc...  AirConditioner   
3  #AirConditioner.airConditionerMode_setAirCondi...  AirConditioner   
4  #AirConditioner.airConditionerMode_setTemperature  AirConditioner   
5                      #AirConditioner.switch_switch  AirConditioner   
6                          #AirConditioner.switch_on  AirConditioner   
7                         #AirConditioner.switch_off  AirConditioner   
8                      #AirConditioner.switch_toggle  AirConditioner   
9  #AirPurifier.airPurifierFanMode_airPurifierFan...     AirPurifier   

                skill                 action      type  \
0  airConditionerMode     airConditionerMode     value   
1  airConditionerMode      targetTemperature     value   
2  airConditionerMode       suppo

In [17]:
import json

def convert_excel_to_device_json(excel_path: str, json_path: str) -> dict:
    df = pd.read_excel(excel_path)
    structured = {}

    for _, row in df.iterrows():
        device = row["device"]
        service_name = f"{row['skill']}_{row['action']}"

        if device not in structured:
            structured[device] = {}

        entry = {}
        for k, v in row.items():
            if pd.notna(v) and k not in ["device", "skill", "action", "full_name"]:
                entry[k] = v

        structured[device][service_name] = entry

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(structured, f, indent=2, ensure_ascii=False)

    return structured


In [18]:
compact = convert_excel_to_device_json("device_skill_action_registry_output.xlsx", "device_services_list_compact.json")

In [19]:
compact

{'AirConditioner': {'airConditionerMode_airConditionerMode': {'type': 'value',
   'descriptor': 'Allows for the control of the air conditioner.',
   'return_descriptor': 'Current mode of the air conditioner',
   'return_type': 'ENUM',
   'enums_descriptor': '• auto - auto\n                • cool - cool\n                • heat - heat'},
  'airConditionerMode_targetTemperature': {'type': 'value',
   'descriptor': 'Allows for the control of the air conditioner.',
   'return_descriptor': 'Current temperature status of the air conditioner',
   'return_type': 'INTEGER',
   'enums_descriptor': '• auto - auto\n                • cool - cool\n                • heat - heat'},
  'airConditionerMode_supportedAcModes': {'type': 'value',
   'descriptor': 'Allows for the control of the air conditioner.',
   'return_descriptor': 'Supported states for this air conditioner to be in',
   'return_type': 'LIST',
   'enums_descriptor': '• auto - auto\n                • cool - cool\n                • heat - h

In [47]:
import json
import re
import pandas as pd

def clean_descriptor_string(desc):
    if not isinstance(desc, str):
        return desc
    parts = re.split(r'\n\s*•', desc.strip())
    if len(parts) == 1:
        return parts[0].strip()
    return [f"• {p.strip()}" if not p.strip().startswith("•") else p.strip() for p in parts]

def convert_excel_to_device_json_cleaned(excel_path: str, json_path: str) -> dict:
    df = pd.read_excel(excel_path)
    structured = {}
    enum_cache = {}

    for _, row in df.iterrows():
        device = row["device"]
        skill = row["skill"]
        service_name = f"{skill}_{row['action']}"

        if device not in structured:
            structured[device] = {}

        entry = {}
        for k, v in row.items():
            if pd.notna(v) and k not in ["device", "skill", "action", "full_name"]:
                cleaned = clean_descriptor_string(v) if 'descriptor' in k else v
                entry[k] = cleaned

        # enums_descriptor 중복 제거 및 skill별 상위 이동 처리
        enum_key = f"{device}::{skill}"
        if "enums_descriptor" in entry:
            if enum_key not in enum_cache:
                enum_cache[enum_key] = entry["enums_descriptor"]
            elif enum_cache[enum_key] == entry["enums_descriptor"]:
                entry.pop("enums_descriptor", None)

        if entry.get("return_type") == "LIST" and entry.get("return_type"):
            entry["return_format"] = f"LIST[{entry['return_type']}]"
        
        if entry.get("argument_type") == "LIST" and entry.get("argument_type"):
            entry["argument_format"] = f"LIST[{entry['return_type']}]"

        structured[device][service_name] = entry

    # 상위 skill 기준으로 __enums__ 항목 추가
    """
    for enum_key, enum_value in enum_cache.items():
        device, skill = enum_key.split("::")
        structured[device][f"{skill}.__enums__"] = {"enums_descriptor": enum_value}
    """
    
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(structured, f, indent=2, ensure_ascii=False)

    return structured


In [48]:
compact = convert_excel_to_device_json_cleaned("device_skill_action_registry_output.xlsx", "device_services_list_compact_cleaned.json")


In [49]:
compact

{'AirConditioner': {'airConditionerMode_airConditionerMode': {'type': 'value',
   'descriptor': 'Allows for the control of the air conditioner.',
   'return_descriptor': 'Current mode of the air conditioner',
   'return_type': 'ENUM',
   'enums_descriptor': ['• auto - auto', '• cool - cool', '• heat - heat']},
  'airConditionerMode_targetTemperature': {'type': 'value',
   'descriptor': 'Allows for the control of the air conditioner.',
   'return_descriptor': 'Current temperature status of the air conditioner',
   'return_type': 'INTEGER'},
  'airConditionerMode_supportedAcModes': {'type': 'value',
   'descriptor': 'Allows for the control of the air conditioner.',
   'return_descriptor': 'Supported states for this air conditioner to be in',
   'return_type': 'LIST',
   'return_format': 'LIST[LIST]'},
  'airConditionerMode_setAirConditionerMode': {'type': 'function',
   'descriptor': 'Allows for the control of the air conditioner.',
   'argument_descriptor': 'Set the air conditioner mode

# service_list_ver.1.5.3 만들기
예시 ~ descriptor 앞으로 + 리스트화

In [ ]:
# 예시 ~ descriptor 앞으로 + 리스트화
{
    "AirConditioner": {
      "descriptor": ["Allows for the control of the air conditioner.", "Set the air conditioner mode"],
      "airConditionerMode_airConditionerMode": {
        "type": "value",
        "return_descriptor": "Current mode of the air conditioner",
        "return_type": "ENUM",
        "enums_descriptor": [
          "• auto - auto",
          "• cool - cool",
          "• heat - heat"
        ]
      },
      "airConditionerMode_targetTemperature": {
        "type": "value",
        "return_descriptor": "Current temperature status of the air conditioner",
        "return_type": "INTEGER"
      },
      "airConditionerMode_supportedAcModes": {
        "type": "value",
        "return_descriptor": "Supported states for this air conditioner to be in",
        "return_type": "LIST",
        "return_format": "LIST[ENUM]"
      },
      "airConditionerMode_setAirConditionerMode": {
        "type": "function",
        "argument_descriptor": "Set the air conditioner mode",
        "argument_type": "ENUM",
        "argument_bounds": "Set the air conditioner mode",
        "return_type": "VOID"
      },
      "airConditionerMode_setTemperature": {
        "type": "function",
        "argument_descriptor": "Set the air conditioner temperature",
        "argument_type": "INTEGER",
        "argument_bounds": "Set the air conditioner temperature",
        "return_type": "VOID"
      },
      "switch_switch": {
        "type": "value",
        "return_descriptor": "A string representation of whether the switch is on or off",
        "return_type": "ENUM",
        "enums_descriptor": [
          "• on - The value of the ``switch`` attribute if the switch is on",
          "• off - The value of the ``switch`` attribute if the switch is off"
        ]
      },
      "switch_on": {
        "type": "function",
        "argument_descriptor": "Turn a switch on",
        "return_type": "VOID"
      },
      "switch_off": {
        "type": "function",
        "argument_descriptor": "Turn a switch off",
        "return_type": "VOID"
      },
      "switch_toggle": {
        "type": "function",
        "argument_descriptor": "Toggle a switch",
        "return_type": "VOID"
      }
    },

# service_list_ver.1.5.4 만들기
type: value/function을 상위 계층으로 하여 구조 개선